# Lezione 13 - Memoria dell'Agente


## Configurazione

Questo notebook dimostra come costruire un agente di prenotazione viaggi con **memoria persistente** utilizzando il **Microsoft Agent Framework** (MAF).

Imparerai come diversi tipi di memoria dell'agente — di lavoro, a breve termine e a lungo termine — influenzano il modo in cui un agente conserva e utilizza le informazioni nelle conversazioni.

**Prerequisiti:**
- Un progetto Azure AI Foundry con un modello di chat distribuito (ad es. `gpt-4o-mini`).
- Accesso effettuato con Azure CLI — esegui `az login` nel tuo terminale.
- `AZURE_AI_PROJECT_ENDPOINT` — l'endpoint del tuo progetto Azure AI Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — il nome del modello distribuito.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Tipi di Memoria degli Agenti

Gli agenti AI possono utilizzare diversi tipi di memoria, ognuno con uno scopo distinto:

### Memoria di Lavoro
Il filo della conversazione stesso — i messaggi scambiati in una singola sessione. L'agente può fare riferimento ai messaggi precedenti nello stesso filo per mantenere la coerenza. In MAF questo viene creato tramite **`agent.create_session()`**, che restituisce un `AgentSession`.

### Memoria a Breve Termine
Informazioni che persistono per la durata di un compito o di una sessione ma non sono archiviate permanentemente. Per esempio, l'agente può accumulare fatti durante una conversazione di pianificazione a più turni e usarli per produrre un itinerario finale.

### Memoria a Lungo Termine
Preferenze e fatti che persistono **tra le sessioni**. Un utente che ritorna non dovrebbe dover ripetere le proprie restrizioni alimentari o lo stile di viaggio. La memoria a lungo termine è tipicamente supportata da un archivio esterno — un database, un file o un indice vettoriale — e resa disponibile all'agente tramite strumenti.


## Memoria di lavoro con le sessioni

La forma più semplice di memoria è la sessione di conversazione. Quando si passa la stessa sessione (creata tramite `agent.create_session()`) a successive chiamate `agent.run()`, l'agente vede l'intera cronologia di quella conversazione e può richiamare dettagli precedenti.

Creiamo un agente di viaggio e dimostriamo la memoria di lavoro.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

L'agente ha ricordato correttamente il budget perché entrambi i messaggi condividono la stessa sessione. Questa è la **memoria di lavoro** — esiste solo per la durata della sessione.

### Cosa succede con un nuovo thread?

Se creiamo una sessione **nuova**, l'agente non ha memoria della conversazione precedente:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Pattern di Memoria a Lungo Termine

Per ricordare le preferenze dell'utente **tra le sessioni**, abbiamo bisogno di un archivio persistente che esista al di fuori del thread di conversazione. L'agente accede a questo archivio tramite **strumenti** — funzioni che può chiamare per salvare e recuperare informazioni.

Di seguito implementiamo un semplice archivio di preferenze in memoria (in produzione lo si supporterebbe con un database o un indice vettoriale) e lo esponiamo come strumenti che l'agente può utilizzare.

### Architettura
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — Utente che prenota per la prima volta un viaggio per un anniversario

Sarah visita per la prima volta. L’agente dovrebbe memorizzare le sue preferenze tramite gli strumenti e usarle per consigliare hotel.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenario 2 — Sarah ritorna dopo settimane

Sarah avvia un **nuovo thread** (simulando una nuova sessione). La memoria di lavoro è vuota, ma l'archivio delle preferenze a lungo termine contiene ancora le sue informazioni. L'agente dovrebbe recuperarle e usarle per personalizzare le raccomandazioni.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Riepilogo

In questa lezione hai imparato tre tipi di memoria per agenti e come implementarli con il Microsoft Agent Framework:

| Tipo di Memoria | Meccanismo MAF | Durata |
|---|---|---|
| **Operativa** | `agent.create_session()` | Singola conversazione |
| **A breve termine** | Contesto accumulato all'interno di un thread | Singolo compito / sessione |
| **A lungo termine** | Archivio esterno accessibile tramite funzioni `@tool` | Tra le sessioni |

### Punti Chiave
1. **`agent.create_session()`** fornisce la memoria operativa — l'agente vede l'intera cronologia della conversazione all'interno di una sessione.
2. **Le nuove sessioni perdono il contesto** — senza memoria a lungo termine l'agente non può ricordare conversazioni passate.
3. **Le funzioni `@tool`** colmano il divario — permettono all'agente di salvare e recuperare informazioni da un archivio persistente.
4. **La personalizzazione migliora nel tempo** — più preferenze sono memorizzate, migliori sono le raccomandazioni dell'agente.

### Applicazioni nel Mondo Reale
- **Assistenza Clienti**: Ricordare la storia e le preferenze del cliente
- **Assistenti Personali**: Mantenere il contesto per giorni o settimane
- **Sanità**: Tracciare informazioni e preferenze del paziente
- **E-commerce**: Shopping personalizzato basato sulla cronologia

### Passi Successivi
- Sostituire il dizionario in memoria con un database o un archivio vettoriale (es. Azure AI Search)
- Aggiungere la scadenza della memoria per informazioni sensibili al tempo
- Costruire sistemi multi-agente con memoria condivisa
- Esplorare il notebook Cognee per una memoria supportata da grafi di conoscenza


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Dichiarazione di non responsabilità**:  
Questo documento è stato tradotto utilizzando il servizio di traduzione automatica [Co-op Translator](https://github.com/Azure/co-op-translator). Pur impegnandoci per garantire la correttezza, si prega di notare che le traduzioni automatiche possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un traduttore umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
